# Panopticon Protocol v3 - Submitted Colab Notebook (Training + Plots)

This notebook keeps the historical filename `Panopticon_Plots_Colab.ipynb` because the already-submitted Google Colab URL cannot be changed. For judge convenience, it now contains the full end-to-end training pipeline from `Panopticon_Training_FINAL.ipynb`, including artifact upload and plot generation.

- **Canonical source notebook in the repo:** `Panopticon_Training_FINAL.ipynb`
- **Compatibility goal of this notebook:** make the submitted Colab URL show a real training script, not only the plotting sidecar
- **If you only want plots:** you can skip directly to the later plot-generation cells after the training/evaluation setup


## Public proof links for judges

If you opened this notebook from the submitted Colab link, here is the public evidence trail for the final training run:

- **Environment Space:** [panopticon-protocol-v3](https://huggingface.co/spaces/Ayush-Kumar0207/panopticon-protocol-v3)
- **Submitted trainer Space:** [panopticon-trainer](https://huggingface.co/spaces/Ayush-Kumar0207/panopticon-trainer)
- **Model repo with final merged weights:** [panopticon-argus-qwen-1.5B](https://huggingface.co/Ayush-Kumar0207/panopticon-argus-qwen-1.5B)
- **Uploaded training metrics:** [training_metrics/](https://huggingface.co/Ayush-Kumar0207/panopticon-argus-qwen-1.5B/tree/main/training_metrics)
- **Uploaded benchmark JSON:** [evaluationResults.json](https://huggingface.co/Ayush-Kumar0207/panopticon-argus-qwen-1.5B/blob/main/evaluationResults.json)
- **Uploaded showcase JSON:** [showcaseResults.json](https://huggingface.co/Ayush-Kumar0207/panopticon-argus-qwen-1.5B/blob/main/showcaseResults.json)
- **README with full training story:** [README.md](https://github.com/Ayush-Kumar0207/panopticon-protocol-v3/blob/main/README.md)
- **Blog mirror used in the submission:** [blog.md](https://huggingface.co/spaces/Ayush-Kumar0207/panopticon-protocol-v3/blob/main/blog.md)

This notebook contains the actual end-to-end training flow below, not just plot regeneration.


In [ ]:
# Minimal training command used by the worker setup
#
# Official worker run (already completed on A10G):
# python train_trl_v2.py --curriculum --episodes 50 --merge
# python generate_plots.py
# python full_evaluation.py --model /data/panopticon-ep50-v2/merged_model --episodes 3 --output evaluationResults.json --plot-dir plots/evaluation --showcase-output showcaseResults.json
#
# This submitted notebook defaults to a fast smoke test so you can verify train -> merge -> plots -> eval locally before trusting the long run.


### Run mode

This notebook now defaults to **`RUN_MODE = "smoke"`** so judges or teammates can quickly verify the full pipeline on Colab without waiting for the 50-episode curriculum.

- `smoke`: runs a tiny easy-level training pass, merges, generates plots, runs a 1-episode evaluation, and **skips uploads**
- `full`: runs the original 50-episode curriculum, generates final plots, runs the 3-episode benchmark, and uploads artifacts

To switch back to the official end-to-end run, change `RUN_MODE` to `"full"` in the configuration cell below.

# 🧠 Panopticon Protocol v3 — Production Training Notebook


---
## 1️⃣ Install Dependencies

In [ ]:
!pip uninstall -y torchao || true
!pip install -q \
  pydantic==2.6.1 \
  gymnasium==0.29.1 \
  numpy==1.26.4 \
  transformers==4.46.3 \
  tokenizers==0.20.3 \
  trl==0.12.2 \
  peft==0.12.0 \
  accelerate==0.34.2 \
  datasets==3.1.0 \
  huggingface-hub==0.26.5 \
  safetensors==0.4.5 \
  matplotlib==3.9.2 \
  fastapi httpx tqdm


---
## 2️⃣ HuggingFace Authentication

Add your HF token as a Colab Secret named `HF_TOKEN` (click the key icon in the sidebar), or paste it when prompted. Use a **write** token if you want to upload the merged model and artifacts back to the Hub.


In [ ]:
import os
from getpass import getpass
from huggingface_hub import login

HF_TOKEN = None
try:
    from google.colab import userdata  # type: ignore
    HF_TOKEN = userdata.get('HF_TOKEN')
except Exception:
    HF_TOKEN = None

if not HF_TOKEN:
    HF_TOKEN = getpass('Enter your HF_TOKEN: ').strip()
    if not HF_TOKEN:
        raise ValueError('HF_TOKEN is required to push artifacts to Hugging Face Hub.')

USERNAME = 'Ayush-Kumar0207'
os.environ['HF_TOKEN'] = HF_TOKEN
login(token=HF_TOKEN, add_to_git_credential=True)
print(f'✅ Successfully logged into Hugging Face as {USERNAME}')


---
## 3️⃣ Clone Repository & Verify

In [ ]:
# CRITICAL: Clone the CORRECT repository
!rm -rf /content/panopticon-protocol-v3
!git clone https://github.com/Ayush-Kumar0207/panopticon-protocol-v3.git /content/panopticon-protocol-v3
%cd /content/panopticon-protocol-v3

# Verify key files exist
import os
required = ['environment.py', 'models.py', 'grader.py', 'train_trl_v2.py']
for f in required:
    status = '✅' if os.path.exists(f) else '❌ MISSING'
    print(f'  {status} {f}')

---
## 4️⃣ Smoke Test — Verify Environment Works

In [ ]:
%cd /content/panopticon-protocol-v3
!python smoke_test.py

---
## 5️⃣ Verify GPU Available

In [ ]:
import torch
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
    print('Recommended for the full judged run: A10G / L4 / better')
else:
    print('No GPU detected. For the full 50-episode run, switch to an A10G/L4 runtime.')

RUN_MODE = "smoke"  # change to "full" for the 50-episode curriculum
MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

if RUN_MODE == "smoke":
    TRAIN_ARGS = f"--level easy --model {MODEL_NAME} --episodes 2 --merge"
    EVAL_EPISODES = 1
    EVAL_PLOT_DIR = "plots/evaluation_smoke"
    DO_UPLOAD = False
else:
    TRAIN_ARGS = f"--curriculum --model {MODEL_NAME} --episodes 50 --merge"
    EVAL_EPISODES = 3
    EVAL_PLOT_DIR = "plots/evaluation"
    DO_UPLOAD = True

print(f"RUN_MODE={RUN_MODE}")
print(f"TRAIN_ARGS={TRAIN_ARGS}")
print(f"EVAL_EPISODES={EVAL_EPISODES}")
print(f"EVAL_PLOT_DIR={EVAL_PLOT_DIR}")
print(f"DO_UPLOAD={DO_UPLOAD}")


---
## 6️⃣ Full Curriculum Training (Easy → Manchurian)

This is the judged path: 50 expert episodes per level, merged final model, and a saved raw log for later plotting.

**Estimated time on A10G small:** ~4–6.5 hours end to end.

**Quick smoke test:** use the optional cell at the bottom if you only want to verify wiring on a smaller GPU.


In [ ]:
%cd /content/panopticon-protocol-v3

import subprocess

train_cmd = f"python train_trl_v2.py {TRAIN_ARGS} 2>&1 | tee output_logs.txt"
print(train_cmd)
subprocess.run(train_cmd, shell=True, check=True)


---
## 7️⃣ Generate Training Plots & Evaluate the Merged Model

This produces the same training figures used in the README and runs the official `full_evaluation.py` benchmark on the merged model.


In [ ]:
%cd /content/panopticon-protocol-v3

import subprocess

subprocess.run("python generate_plots.py", shell=True, check=True)
eval_cmd = f"python full_evaluation.py --model merged_model --episodes {EVAL_EPISODES} --output evaluationResults.json --plot-dir {EVAL_PLOT_DIR} --showcase-output showcaseResults.json"
print(eval_cmd)
subprocess.run(eval_cmd, shell=True, check=True)


---
## 8️⃣ Push Model + Artifacts to HuggingFace Hub


In [ ]:
from pathlib import Path

if not DO_UPLOAD:
    print("Smoke mode enabled: skipping Hugging Face uploads. Review local output_logs.txt, plots/, evaluationResults.json, and showcaseResults.json first.")
else:
    from huggingface_hub import HfApi

    api = HfApi(token=HF_TOKEN)
    repo_id = f'{USERNAME}/panopticon-argus-qwen-1.5B'

    api.create_repo(repo_id, repo_type='model', exist_ok=True)
    print(f'Created/verified repository: https://huggingface.co/{repo_id}')

    merged_model = Path('merged_model')
    if not merged_model.exists():
        raise FileNotFoundError('merged_model not found. Make sure training ran with --merge.')

    api.upload_folder(
        folder_path=str(merged_model),
        repo_id=repo_id,
        repo_type='model',
    )
    print('✅ Uploaded merged model folder.')

    for filename in ['evaluationResults.json', 'showcaseResults.json', 'output_logs.txt']:
        path = Path(filename)
        if path.exists():
            api.upload_file(
                path_or_fileobj=str(path),
                path_in_repo=filename if filename != 'output_logs.txt' else 'training_metrics/output_logs.txt',
                repo_id=repo_id,
                repo_type='model',
            )
            print(f'✅ Uploaded {filename}')

    plots_dir = Path('plots')
    if plots_dir.exists():
        api.upload_folder(
            folder_path=str(plots_dir),
            path_in_repo='plots',
            repo_id=repo_id,
            repo_type='model',
        )
        print('✅ Uploaded plots/')

    print(f'Final artifacts available at https://huggingface.co/{repo_id}')


---
## 9️⃣ (Optional) Quick Single-Level Test

Fast test on one level (~20 min) before committing to full curriculum:

### Run-mode reminder

The active `RUN_MODE` is configured earlier in the notebook, right after the GPU check.

- Leave it as `"smoke"` to quickly verify train -> merge -> plots -> eval without uploads.
- Change it to `"full"` there when you want the full 50-episode curriculum and artifact upload path.


---
## 📎 Links

- **Demo Space**: [panopticon-protocol-v3](https://huggingface.co/spaces/Ayush-Kumar0207/panopticon-protocol-v3)
- **Submitted Trainer Space**: [panopticon-trainer](https://huggingface.co/spaces/Ayush-Kumar0207/panopticon-trainer)
- **Model Repo**: [panopticon-argus-qwen-1.5B](https://huggingface.co/Ayush-Kumar0207/panopticon-argus-qwen-1.5B)
- **GitHub**: [panopticon-protocol-v3](https://github.com/Ayush-Kumar0207/panopticon-protocol-v3)

*Meta PyTorch OpenEnv Hackathon x Scaler Grand Finale, April 2026*

*Team: Ayush Kumar & Ravi Prashant*
